# 모델을 정식 서비스로 배포하기 (TorchServe 패키징)

앞서 만든 모델을 **서빙 서버가 요구하는 표준 형식(.mar)** 으로 포장하고,
사내 모델 저장소(MinIO)에 등록해 API 서비스로 띄웁니다.

> `dm_deploy.ipynb`의 2번 셀까지 실행되어 `deploy/dm-classifier/1/model.pt`가 있어야 합니다.

## 1. 패키징 도구 설치

In [ ]:
!pip install -q torch-model-archiver 2>&1 | tail -2
print("설치 완료")

## 2. 추론 처리기(handler) 작성

서빙 서버가 이미지를 받아 어떻게 처리할지 정의합니다.
(이미지 → 128×128 변환 → 모델 추론 → OK/FAIL + 확신도 반환)

In [ ]:
handler_code = r'''
import io, json, base64, torch
from PIL import Image
from torchvision import transforms
from ts.torch_handler.base_handler import BaseHandler


class DMHandler(BaseHandler):
    """DataMatrix 판독 가능성 분류 핸들러"""

    def initialize(self, ctx):
        super().initialize(ctx)
        self.tf = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.classes = ["OK", "FAIL"]

    def preprocess(self, data):
        imgs = []
        for row in data:
            raw = row.get("data") or row.get("body")
            if isinstance(raw, (bytes, bytearray)):
                try:
                    payload = json.loads(raw)
                    raw = base64.b64decode(payload["instances"][0]["image_b64"])
                except Exception:
                    pass
            elif isinstance(raw, dict):
                raw = base64.b64decode(raw["instances"][0]["image_b64"])
            img = Image.open(io.BytesIO(raw)).convert("RGB")
            imgs.append(self.tf(img))
        return torch.stack(imgs).to(self.device)

    def inference(self, x):
        with torch.no_grad():
            return torch.softmax(self.model(x), 1)

    def postprocess(self, out):
        res = []
        for p in out:
            i = int(p.argmax())
            res.append({"prediction": self.classes[i],
                        "confidence": round(float(p[i]) * 100, 1),
                        "probabilities": {"OK": round(float(p[0]) * 100, 1),
                                          "FAIL": round(float(p[1]) * 100, 1)}})
        return res
'''

with open("dm_handler.py", "w", encoding="utf-8") as f:
    f.write(handler_code)
print("dm_handler.py 작성 완료")

## 3. .mar 패키지 생성

In [ ]:
import os, shutil

os.makedirs("serve_pkg/model-store", exist_ok=True)
os.makedirs("serve_pkg/config", exist_ok=True)

!torch-model-archiver --model-name dm-classifier --version 1.0 \
  --serialized-file deploy/dm-classifier/1/model.pt \
  --handler dm_handler.py \
  --export-path serve_pkg/model-store --force

# TorchServe 설정 파일
cfg = '''inference_address=http://0.0.0.0:8085
management_address=http://0.0.0.0:8085
metrics_address=http://0.0.0.0:8082
enable_envvars_config=true
install_py_dep_per_model=true
model_store=/mnt/models/model-store
load_models=all
'''
with open("serve_pkg/config/config.properties", "w") as f:
    f.write(cfg)

mb = os.path.getsize("serve_pkg/model-store/dm-classifier.mar") / 1024 / 1024
print(f"패키지 생성 완료: dm-classifier.mar ({mb:.1f} MB)")
!ls -R serve_pkg

## 4. 모델 저장소에 등록

포털 백엔드를 통해 사내 모델 저장소(MinIO)에 업로드합니다.

In [ ]:
import requests, os

CMP_API = "http://192.168.0.41:8000"   # 포털 백엔드

r = requests.post(f"{CMP_API}/api/v1/models/import", json={
    "notebook": "dm-train",
    "path": "serve_pkg",
    "modelName": "dm-classifier-serve",
    "namespace": "ai-dev-team",
}, timeout=600)
print(r.status_code, r.json())

## 5. 배포 확인

포털 → **AI 개발환경 → Serving** 탭에서 `dm-classifier` 서비스가 Ready 로 바뀌면 완료입니다.
Ready가 되면 아래 셀로 실제 호출해 봅니다.

In [ ]:
import base64, json, requests
from pathlib import Path

CMP_API = "http://192.168.0.41:8000"
img_path = sorted(Path("qr_crops_failed").glob("*.jpg"))[0]
b64 = base64.b64encode(img_path.read_bytes()).decode()

r = requests.post(f"{CMP_API}/api/v1/serving/dm-classifier/predict",
                  json={"instances": [{"image_b64": b64}]}, timeout=60)
print("HTTP", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))
print()
print("테스트 이미지:", img_path.name, "(정답: FAIL)")

## 6. 정리

이제 이 모델은 **포털에 등록된 정식 서비스**입니다.

- 모델 저장소: `s3://models/dm-classifier-serve`
- 호출 주소: `POST /api/v1/serving/dm-classifier/predict`
- 다른 시스템(현장 검사 PC, 웹 화면 등)에서 이 주소로 이미지를 보내면 판정 결과를 받습니다.